# City of Boston Property Assessment Data

The goal of this poject is to answer the following questions:
- Does owner occupancy decline after the '08 financial crash and if so, was it everywhere in Boston or only certain neighborhoods/zip codes? 
- Also, was this accelerating an already occurring trend or reversing a more equitable division of wealth? 
- Also also regarding owners: what is the breakdown between owned by a person, owned by a trust, and owned by a company (and then further breaking down the kinds of companies)?

Questions after seeing the data:
- What is the worth of a square ft in somewhere like Beacon Hill versus somewhere like Mattapan?
- What is the ratio of owner occupied units in a places like Beacon Hill versus places like Mattapan?
- How many of out-of-state landlords are there?
- Adding in utility data, can we see which housing units are not occupied?

Data Source: https://data.boston.gov/dataset/property-assessment

Issues to resolve:
    X drop unneeded cols
    X add column for FY
    XX rename columns
    - fix known issues:
        - convert zip code to int
            -zip code
            -mailing zip code
        - add back the dropped leading 0 to the zips
        - move the decimal in gross_tax two places to the left
        - do lot size/land sf and gross area need decimal moved?  
        O separate mailing city state column into mailing city and mailing state columns
            - add a country column
            - fix typos
            - separate out zip codes
    - concat all dfs together

In [211]:
import pandas as pd
import numpy as np
import datetime as dt
import os
from sqlalchemy import create_engine

### Figure out which columns need to be dropped

In [17]:
# grab all file names in directory
cobDirectory = os.listdir()
cobCSVDirectory = []

#pare the list down to only the property assessment files
for item in cobDirectory:
    if '.csv' in item:
        cobCSVDirectory.append(item)
        
fullColNameList = []

#create a list of all unique column names, sorted alphabetically
for e in cobCSVDirectory:
    colNameList = list(pd.read_csv(e, nrows=1))
    for element in colNameList:
        fullColNameList.append(element)
    
fullColNameList = list(set(fullColNameList))

fullColNameList.sort()

print(fullColNameList)

[' GROSS_TAX ', 'AC_TYPE', 'AV_BLDG', 'AV_LAND', 'AV_TOTAL', 'BDRM_COND', 'BED_RMS', 'BLDG_AREA', 'BLDG_SEQ', 'BLDG_TYPE', 'BLDG_VALUE', 'BTHRM_STYLE1', 'BTHRM_STYLE2', 'BTHRM_STYLE3', 'CD_FLOOR', 'CITY', 'CM_ID', 'COM_UNITS', 'CORNER_UNIT', 'EXT_COND', 'EXT_FINISHED', 'EXT_FNISHED', 'FIREPLACES', 'FIRE_PLACE', 'FULL_BTH', 'FY2004_BLDG', 'FY2004_TOTAL', 'FY2006_BLDG', 'FY2006_LAND', 'FY2006_TOTAL', 'FY2007_BLDG', 'FY2007_LAND', 'FY2007_TOTAL', 'FY2008_BLDG', 'FY2008_GROSS_TAX', 'FY2008_LAND', 'FY2008_TOTAL', 'FY200_ LAND', 'GIS_ID', 'GROSS_AREA', 'GROSS_TAX', 'HEAT_FUEL', 'HEAT_SYSTEM', 'HEAT_TYPE', 'HLF_BTH', 'INT_COND', 'INT_WALL', 'KITCHEN', 'KITCHENS', 'KITCHEN_STYLE1', 'KITCHEN_STYLE2', 'KITCHEN_STYLE3', 'KITCHEN_TYPE', 'LAND_SF', 'LAND_VALUE', 'LATITUDE', 'LIVING _AREA', 'LIVING_AREA', 'LONGITUDE', 'LOTSIZE', 'LU', 'LUC', 'LU_DESC', 'Location', 'MAIL CS', 'MAIL_ADDRESS', 'MAIL_ADDRESSEE', 'MAIL_CITY', 'MAIL_CITY_STATE', 'MAIL_CS', 'MAIL_STATE', 'MAIL_STREET_ADDRESS', 'MAIL_ZIP', 

In [18]:
#compared column names with property assessment data key PDFs to see which columns need to be kept and which to group together
# 'LUC' stands for Land Use Code and the code descriptions don't 100% align with the LU_DESC values but the LU_DESC values based on spot checking are more detailed
# 'SFYI_VALUE' is not listed as a data key anywhere, but may stand for Special Features Yard Items Value
# GROSS_TAX col note for some years: Amount is based upon the total assessed value multiplied by the tax rate for the given year. The gross tax amount does not include any personal exemptions.  The value is stored in a non-decimal format.  Hence, $1,654.23 is displayed as 165423 .
colsToBeKept = ['OWNER_MAIL_CS', 'Owner_MAIL_CS', 'MAIL_CITY_STATE',  'MAIL_CS', 'MAIL CS', 'BLDG_TYPE', 'Fiscal_Year',
                'PID',  'Parcel_ID', 'CM_ID', 'GIS_ID', 'UNIT_NUM', 'CITY', 'LU', 'LU_DESC', 'OWN_OCC',
               'ST_NUM_CHAR', 'ST_NUM', 'ST_NAME', 'ST_NAME_SFX', 'ST_NAME_SUF', 'ST_SFX', 'ZIPCODE','ZIP_CODE',
               'OWNER', 'OWNER FY04', 'OWNER FY07', 'MAIL_ADDRESSEE', 'MAIL_CITY', 'MAIL_STATE', 'LOTSIZE','LAND_SF',
               'MAIL_STREET_ADDRESS','Owner_MAIL_ADDRESS', 'OWNER MAIL ADDRESS', 'OWNER_MAIL_ADDRESS', 'MAIL_ADDRESS',
               'MAIL_ZIP', 'MAIL_ZIPCODE', 'MAIL_ZIP_CODE',  'OWNER_MAIL_ZIPCODE', 'Owner_MAIL_ZIPCODE',
               'BLDG_AREA', 'GROSS_AREA', 'LIVING _AREA', 'LIVING_AREA', ' GROSS_TAX ', 'GROSS_TAX', 'FY2008_GROSS_TAX',
               'LAND_VALUE', 'FY200_ LAND', 'FY2006_LAND', 'FY2007_LAND', 'FY2008_LAND', 'AV_LAND',
               'BLDG_VALUE', 'FY2004_BLDG', 'FY2006_BLDG', 'FY2007_BLDG', 'FY2008_BLDG', 'AV_BLDG',
               'TOTAL_VALUE', 'FY2004_TOTAL', 'FY2006_TOTAL', 'FY2007_TOTAL', 'FY2008_TOTAL', 'AV_TOTAL']

### Figure out which columns need to be renamed

In [133]:
#compared column names with property assessment data key PDFs to see which columns need to be grouped together
ownerMailingAddressCityState = ['OWNER_MAIL_CS', 'Owner_MAIL_CS', 'MAIL_CITY_STATE',  'MAIL_CS', 'MAIL CS']

renameCols = {'ParcelID': ['PID',  'Parcel_ID'],
 'CondoMainID': 'CM_ID',
 'GeographicInformationSystemID': 'GIS_ID',
 'StreetNumber': ['ST_NUM_CHAR', 'ST_NUM'],
 'StreetNumber2': 'ST_NUM2',
 'StreetName': 'ST_NAME',
 'StreetSuffix': ['ST_NAME_SFX', 'ST_NAME_SUF', 'ST_SFX'], 
 'UnitNumber': 'UNIT_NUM',
 'City': 'CITY',
 'ZipCode': ['ZIPCODE','ZIP_CODE'],
 'LandUse' : 'LU',
 'LandUseDescription': 'LU_DESC',
 'BuildingType': 'BLDG_TYPE',
 'OwnerOccupied': 'OWN_OCC',
 'Owner': ['OWNER', 'OWNER FY04', 'OWNER FY07'],
 'OwnerMailingAddressAddressee': 'MAIL_ADDRESSEE',
 'OwnerMailingAddressStreet': ['MAIL_STREET_ADDRESS','Owner_MAIL_ADDRESS', 'OWNER MAIL ADDRESS', 'OWNER_MAIL_ADDRESS', 'MAIL_ADDRESS'],
 'OwnerMailingAddressCity': 'MAIL_CITY',
 'OwnerMailingAddressState': 'MAIL_STATE',
 'OwnerMailingAddressZipCode': ['MAIL_ZIP', 'MAIL_ZIPCODE', 'MAIL_ZIP_CODE',  'OWNER_MAIL_ZIPCODE', 'Owner_MAIL_ZIPCODE'],
 'LandSquareFoot': ['LOTSIZE','LAND_SF'],
 'BuildingArea':'BLDG_AREA', 
 'GrossArea': 'GROSS_AREA',
 'LivingArea': ['LIVING _AREA', 'LIVING_AREA'],
 'LandValue': ['LAND_VALUE', 'FY200_ LAND', 'FY2006_LAND', 'FY2007_LAND', 'FY2008_LAND', 'AV_LAND'],
 'BuildingValue': ['BLDG_VALUE', 'FY2004_BLDG', 'FY2006_BLDG', 'FY2007_BLDG', 'FY2008_BLDG', 'AV_BLDG'],
 'TotalValue': ['TOTAL_VALUE', 'FY2004_TOTAL', 'FY2006_TOTAL', 'FY2007_TOTAL', 'FY2008_TOTAL', 'AV_TOTAL'],
 'GrossTax': [' GROSS_TAX ', 'GROSS_TAX', 'FY2008_GROSS_TAX']}

In [134]:
def readInData(fileName, year):
    #read in data
    df = pd.read_csv(fileName)
    #add in fiscal year column
    df['Fiscal_Year'] = year
    #remove unneeded columns
    df = df[df.columns.intersection(colsToBeKept)]
    return df

In [297]:
pa2004 = readInData('data2004-lite.csv', 2004)
pa2005 = readInData('data2005-lite.csv', 2005)
pa2006 = readInData('data2006lite.csv', 2006)
pa2007 = readInData('fy2007.csv', 2007)
pa2008 = readInData('property-assessment-fy08.csv', 2008)
pa2009 = readInData('property-assessment-fy09.csv', 2009)
pa2010 = readInData('property-assessment-fy10.csv', 2010)
pa2011 = readInData('property-assessment-fy11.csv', 2011)
pa2012 = readInData('property-assessment-fy12.csv', 2012)
pa2013 = readInData('property-assessment-fy13.csv', 2013)
pa2014 = readInData('property-assessment-fy2014.csv', 2014)
pa2015 = readInData('property-assessment-fy2015.csv', 2015)
pa2016 = readInData('property-assessment-fy2016.csv', 2016)
pa2017 = readInData('property-assessment-fy2017.csv', 2017)
pa2018 = readInData('ast2018full.csv', 2018)
pa2019 = readInData('fy19fullpropassess.csv', 2019)
pa2020 = readInData('data2020-full.csv', 2020)
pa2021 = readInData('data2021-full.csv', 2021)
pa2022 = readInData('fy2022pa-4.csv', 2022)
pa2023 = readInData('fy2023-property-assessment-data.csv', 2023)
pa2024 = readInData('fy2024-property-assessment-data_1_5_2024.csv', 2024)
pa2025 = readInData('fy2025-property-assessment-data_12_30_2024.csv', 2025)
pa2026 = readInData('fy2026-property-assessment-data_rev.csv', 2026)

/Users/jbryant/opt/anaconda3/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3263: DtypeWarning: Columns (13) have mixed types.Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):
/Users/jbryant/opt/anaconda3/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3263: DtypeWarning: Columns (3) have mixed types.Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):
/Users/jbryant/opt/anaconda3/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3263: DtypeWarning: Columns (6,13) have mixed types.Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):
/Users/jbryant/opt/anaconda3/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3263: DtypeWarning: Columns (13,25,26,27,33,34,37,41,44,45,48,49,50,51,52) have mixed types.Specify dtype option on import or set low_memory=False.
  if (aw

In [151]:
propertyAssessmentList = [pa2004, pa2005, pa2006, pa2007, pa2008, pa2009, pa2010, pa2011, pa2012, pa2013, pa2014, 
                          pa2015, pa2016, pa2017, pa2018, pa2019, pa2020, pa2021, pa2022, pa2023, pa2024, pa2025, 
                          pa2026]

for element in propertyAssessmentList:
    print(element['Fiscal_Year'][0])
    print(element.columns)

2004
Index(['PID', 'CM_ID', 'ST_NUM', 'ST_NAME', 'ST_NAME_SFX', 'UNIT_NUM',
       'ZIPCODE', 'LU', 'OWN_OCC', 'OWNER FY04', 'MAIL_ADDRESS',
       'MAIL_CITY_STATE', 'MAIL_ZIP', 'LOTSIZE', 'GROSS_AREA', 'LIVING _AREA',
       'FY2004_TOTAL', 'FY200_ LAND', 'FY2004_BLDG', 'GROSS_TAX',
       'Fiscal_Year'],
      dtype='object')
2005
Index(['PID', 'CM_ID', 'ST_NUM', 'ST_NAME', 'ST_NAME_SFX', 'UNIT_NUM',
       'ZIPCODE', 'LU', 'OWN_OCC', 'OWNER FY04', 'MAIL_ADDRESS',
       'MAIL_CITY_STATE', 'MAIL_ZIP', 'LOTSIZE', 'GROSS_AREA', 'LIVING _AREA',
       'FY2004_TOTAL', 'FY200_ LAND', 'FY2004_BLDG', 'GROSS_TAX',
       'Fiscal_Year'],
      dtype='object')
2006
Index(['PID', 'CM_ID', 'OWNER', 'ST_NUM', 'ST_NAME', 'ST_SFX', 'UNIT_NUM',
       'ZIPCODE', 'LU', 'LOTSIZE', 'MAIL_ADDRESS', 'MAIL_CS', 'MAIL_ZIP',
       'OWN_OCC', 'GROSS_AREA', 'FY2006_TOTAL', 'FY2006_LAND', 'FY2006_BLDG',
       'GROSS_TAX', 'Fiscal_Year'],
      dtype='object')
2007
Index(['PID', 'CM_ID', 'OWNER FY07', 'ST_NU

In [299]:
pa2019['UNIT_NUM']

0         2-F
1         2-R
2         3-F
3         3-R
4           4
         ... 
174663    NaN
174664    NaN
174665    NaN
174666    NaN
174667    NaN
Name: UNIT_NUM, Length: 174668, dtype: object

In [136]:
def renameColumns(df):
    for col in df:
        for key, value in renameCols.items():
            if str(type(value)) == "<class 'str'>":
                if col == value:
                    df.rename(columns={col: key}, inplace=True)
            elif str(type(value)) == "<class 'list'>":
                if col in value:
                    df.rename(columns={col: key}, inplace=True)
    return df

In [308]:
pa2004 = renameColumns(pa2004)
pa2005 = renameColumns(pa2005)
pa2006 = renameColumns(pa2006)
pa2007 = renameColumns(pa2007)
pa2008 = renameColumns(pa2008)
pa2009 = renameColumns(pa2009)
pa2010 = renameColumns(pa2010)
pa2011 = renameColumns(pa2011)
pa2012 = renameColumns(pa2012)
pa2013 = renameColumns(pa2013)
pa2014 = renameColumns(pa2014)
pa2015 = renameColumns(pa2015)
pa2016 = renameColumns(pa2016)
pa2017 = renameColumns(pa2017)
pa2018 = renameColumns(pa2018)
pa2019 = renameColumns(pa2019)
pa2020 = renameColumns(pa2020)
pa2021 = renameColumns(pa2021)
pa2022 = renameColumns(pa2022)
pa2023 = renameColumns(pa2023)
pa2024 = renameColumns(pa2024)
pa2025 = renameColumns(pa2025)
pa2026 = renameColumns(pa2026)

In [159]:
propertyAssessmentList = [pa2004, pa2005, pa2006, pa2007, pa2008, pa2009, pa2010, pa2011, pa2012, pa2013, pa2014, 
                          pa2015, pa2016, pa2017, pa2018, pa2019, pa2020, pa2021, pa2022, pa2023, pa2024, pa2025, 
                          pa2026]

for element in propertyAssessmentList:
    print(element['Fiscal_Year'][0])
    print(element.columns)

2004
Index(['ParcelID', 'CondoMainID', 'StreetNumber', 'StreetName', 'StreetSuffix',
       'UnitNumber', 'ZipCode', 'LandUse', 'OwnerOccupied', 'Owner',
       'OwnerMailingAddressStreet', 'MAIL_CITY_STATE',
       'OwnerMailingAddressZipCode', 'LandSquareFoot', 'GrossArea',
       'LivingArea', 'TotalValue', 'LandValue', 'BuildingValue', 'GrossTax',
       'Fiscal_Year'],
      dtype='object')
2005
Index(['ParcelID', 'CondoMainID', 'StreetNumber', 'StreetName', 'StreetSuffix',
       'UnitNumber', 'ZipCode', 'LandUse', 'OwnerOccupied', 'Owner',
       'OwnerMailingAddressStreet', 'MAIL_CITY_STATE',
       'OwnerMailingAddressZipCode', 'LandSquareFoot', 'GrossArea',
       'LivingArea', 'TotalValue', 'LandValue', 'BuildingValue', 'GrossTax',
       'Fiscal_Year'],
      dtype='object')
2006
Index(['ParcelID', 'CondoMainID', 'Owner', 'StreetNumber', 'StreetName',
       'StreetSuffix', 'UnitNumber', 'ZipCode', 'LandUse', 'LandSquareFoot',
       'OwnerMailingAddressStreet', 'MAIL_CS', 

In [168]:
#export the necessary columns to comb through to correct the data
pa2004[['OwnerMailingAddressStreet', 'MAIL_CITY_STATE', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2004.csv')
pa2005[['OwnerMailingAddressStreet', 'MAIL_CITY_STATE', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2005.csv')
pa2006[['OwnerMailingAddressStreet', 'MAIL_CS', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2006.csv')
pa2007[['OwnerMailingAddressStreet', 'MAIL_CS', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2007.csv')
pa2008[['OwnerMailingAddressStreet', 'MAIL CS', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2008.csv')
pa2009[['OwnerMailingAddressStreet', 'MAIL CS', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2009.csv')
pa2010[['OwnerMailingAddressStreet', 'MAIL CS', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2010.csv')
pa2011[['OwnerMailingAddressStreet', 'MAIL CS', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2011.csv')
pa2012[['OwnerMailingAddressStreet', 'MAIL CS', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2012.csv')
pa2013[['OwnerMailingAddressStreet', 'MAIL CS', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2013.csv')
pa2014[['OwnerMailingAddressStreet', 'Owner_MAIL_CS', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2014.csv')
pa2015[['OwnerMailingAddressStreet', 'OWNER_MAIL_CS', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2015.csv')
pa2016[['OwnerMailingAddressStreet', 'MAIL CS', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2016.csv')
pa2017[['OwnerMailingAddressStreet', 'MAIL CS', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2017.csv')
pa2018[['OwnerMailingAddressStreet', 'MAIL CS', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2018.csv')
pa2019[['OwnerMailingAddressStreet', 'MAIL CS', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2019.csv')
pa2020[['OwnerMailingAddressStreet', 'MAIL CS', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2020.csv')
pa2021[['OwnerMailingAddressStreet',  'OwnerMailingAddressCity', 'OwnerMailingAddressState', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2021.csv')
pa2022[['OwnerMailingAddressStreet',  'OwnerMailingAddressCity', 'OwnerMailingAddressState', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2022.csv')
pa2023[['OwnerMailingAddressStreet', 'Fiscal_Year']].to_csv('fy2023.csv')
pa2024[['OwnerMailingAddressStreet',  'OwnerMailingAddressCity', 'OwnerMailingAddressState', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2024.csv')
pa2025[['OwnerMailingAddressStreet',  'OwnerMailingAddressCity', 'OwnerMailingAddressState', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2025.csv')
pa2026[['OwnerMailingAddressStreet',  'OwnerMailingAddressCity', 'OwnerMailingAddressState', 'OwnerMailingAddressZipCode', 'Fiscal_Year']].to_csv('fy2026.csv')

In [253]:
#City State Corrections
pa2019['MAIL CS'].iloc[43] = 'ATLANTA GA'
pa2019['MAIL CS'].iloc[402] = 'MARSHFIELD MA'
pa2019['MAIL CS'].iloc[615] = 'LYNNFIELD MA'
pa2019['MAIL CS'].iloc[760] = 'MELROSE MA'
pa2019['MAIL CS'].iloc[2823] = 'BOSTON MA'
pa2019['MAIL CS'].iloc[3035] = 'OAKLAND CA'
pa2019['MAIL CS'].iloc[3314] = 'NEWTON MA'
pa2019['MAIL CS'].iloc[4047] = 'BOSTON MA'
pa2019['MAIL CS'].iloc[17581] = 'MARSHFIELD MA'
pa2019['MAIL CS'].iloc[21355] = 'WELLINGTON FL'
pa2019['MAIL CS'].iloc[24256] = 'LANTANA FL'
pa2019['MAIL CS'].iloc[26310] = 'INDIAN CREEK FL'
pa2019['MAIL CS'].iloc[64107] = 'TOLEDO OH'
pa2019['MAIL CS'].iloc[68163] = 'ELLSWORTH ME'
pa2019['MAIL CS'].iloc[72803] = 'GILBERT AZ'
pa2019['MAIL CS'].iloc[84955] = 'DURANGO CO'
pa2019['MAIL CS'].iloc[93056] = 'ALBANY NY'
pa2019['MAIL CS'].iloc[93541] = 'BOSTON MA'
pa2019['MAIL CS'].iloc[101686] = 'CHESTERFIELD MO'
pa2019['MAIL CS'].iloc[102112] = 'SUGARLAND TX'
pa2019['MAIL CS'].iloc[111126] = 'OAK BROOK IL'
pa2019['MAIL CS'].iloc[123009] = 'FALMOUTH MA'
pa2019['MAIL CS'].iloc[137696] = 'OAKLAND CA'
pa2019['MAIL CS'].iloc[142310] = 'BUZZARDS BAY MA'
pa2019['MAIL CS'].iloc[148745] = 'HARTFORD CT'
pa2019['MAIL CS'].iloc[150022] = 'NORWELL MA'
pa2019['MAIL CS'].iloc[160691] = 'STERLING VA'
pa2019['MAIL CS'].iloc[163039] = 'QUEENS NY'
pa2019['MAIL CS'].iloc[172524] = 'BELMONT MA'
pa2019['MAIL CS'].iloc[174104] = 'BOSTON MA'
pa2019['MAIL CS'].iloc[31566] = 'BOSTON MA'
pa2019['MAIL CS'].iloc[3925] = 'BOSTON MA'
pa2019['MAIL CS'].iloc[6793] = 'BOSTON MA'
pa2019['MAIL CS'].iloc[7184] = 'BOSTON MA'
pa2019['MAIL CS'].iloc[14330] = 'JAY VT'
pa2019['MAIL CS'].iloc[25533] = 'MURRAY UT'
pa2019['MAIL CS'].iloc[41852] = 'NEW YORK NY'
pa2019['MAIL CS'].iloc[49155] = 'MARINA DEL RAY CA'
pa2019['MAIL CS'].iloc[62541] = 'BRIDGEWATER MA'
pa2019['MAIL CS'].iloc[71489] = 'QUINCY MA'
pa2019['MAIL CS'].iloc[80506] = 'BOSTON MA'
pa2019['MAIL CS'].iloc[88028] = 'BOSTON MA'
pa2019['MAIL CS'].iloc[105960] = 'QUINCY MA'
pa2019['MAIL CS'].iloc[135140] = 'SANDY UT'
pa2019['MAIL CS'].iloc[141675] = 'NORWELL MA'
pa2019['MAIL CS'].iloc[144434] = 'BOSTON MA'
pa2019['MAIL CS'].iloc[146949] = 'BOSTON MA'
pa2019['MAIL CS'].iloc[161462] = 'NORTH READING MA'

#Address Corrections
pa2019['MAIL CS'].iloc[40330] = 'ORLANDO FL'
pa2019['OwnerMailingAddressZipCode'].iloc[40330] = 32839
pa2019['MAIL CS'].iloc[78181] = 'BOSTON MA'
pa2019['OwnerMailingAddressAddressee'].iloc[78181] = 'C/O SAVA KELESIDIS'
pa2019['OwnerMailingAddressStreet'].iloc[78181] = '31 DANA AVE STE 3A'
pa2019['MAIL CS'].iloc[40330] = 'LITHONIA GA'
pa2019['OwnerMailingAddressZipCode'].iloc[40330] = 30038   
pa2019['MAIL CS'].iloc[104175] = 'LITHONIA GA'
pa2019['OwnerMailingAddressZipCode'].iloc[104175] = 30038   
pa2019['OwnerMailingAddressAddressee'].iloc[154339] = ''
pa2019['OwnerMailingAddressStreet'].iloc[154339] = '85 WESTOVER ST'
pa2019['MAIL CS'].iloc[154339] = 'BOSTON MA'
pa2019['MAIL CS'].iloc[158180] = 'FLORENCE SC'
pa2019['OwnerMailingAddressZipCode'].iloc[158180] = 29502  

/Users/jbryant/opt/anaconda3/lib/python3.8/site-packages/pandas/core/indexing.py:671: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_with_indexer(indexer, value)


In [260]:
#to remove extra whitespaces in values
for index in pa2019['MAIL CS'].index:
    value = pa2019['MAIL CS'].iloc[index]
    newValue = " ".join(value.split())
    pa2019['MAIL CS'].iloc[index] = newValue

/Users/jbryant/opt/anaconda3/lib/python3.8/site-packages/pandas/core/indexing.py:671: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_with_indexer(indexer, value)


In [261]:
# create two dfs: one for owners with national address and one for owners with international address
intlList = [5893, 13957, 14100, 14126, 14391, 14799, 16495, 16550, 17566, 17570, 17629, 17659, 17790, 19220, 19267, 
            19702, 19704, 20876, 21432, 21510, 21546, 22347, 23569, 25856, 25933, 25958, 26003, 26173, 26279, 26528,
            26791, 27131, 27282, 27310, 27337, 27369, 27516, 27599, 27676, 28146, 28456, 29132, 32380, 35373, 35500,
            35534, 35564, 35821, 35865, 35988, 36733, 36749, 37099, 39604, 39673, 40001, 41407, 41951, 42329, 42528,
            42919, 43565, 44088, 44167, 44532, 44572, 45700, 46357, 46632, 47437, 48190, 48291, 48517, 48545, 48740,
            50687, 50695, 50825, 50963, 51019, 51075, 51308, 51449, 51525, 51550, 51628, 51833, 51874, 52040, 52326,
            52558, 53072, 55177, 59217, 71816, 72446, 72727, 73075, 73568, 78495, 80405, 93894, 133940, 135234, 
            136684, 137487, 144809, 150151, 153996, 157625, 157915, 159315, 159674, 160059, 160561, 160581, 161128,
            161645, 163446, 163570, 166209, 167899, 169783, 172581, 172916, 173341, 173729, 48716, 20903, 21338, 
            26305, 26360, 26364, 26442, 26782, 35265, 38054, 42682, 50661, 50828, 165063, 173346, 18116, 18164, 
            18212, 18985, 25003, 35900, 48096, 50077, 51737, 52709]
pa2019_intl = pa2019.iloc[intlList]
pa2019_us = pa2019.loc[~pa2019.index.isin(intlList)]

In [262]:
# create function to split the city state column of a corrected data frame subset
#split all columns that contain both city and state
def cityStateSplit(df):
    for column in df:
        if column in ownerMailingAddressCityState:
            df['OwnerMailingAddressCity'] = df[column].str[:-3]
            df['OwnerMailingAddressRegionState'] = df[column].str[-2:]
            df['OwnerMailingAddressCountry'] = 'USA'
        else:
            pass
    return df

In [263]:
#test cityStateSplit function
pa2019_us = cityStateSplit(pa2019_us)

#check that all non-US states and territories have correct abbreviations
set(list(pa2019_us['OwnerMailingAddressRegionState']))

<ipython-input-262-9dc3b34dfc79>:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['OwnerMailingAddressCity'] = df[column].str[:-3]
<ipython-input-262-9dc3b34dfc79>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['OwnerMailingAddressRegionState'] = df[column].str[-2:]
<ipython-input-262-9dc3b34dfc79>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas

{'AK',
 'AL',
 'AR',
 'AZ',
 'CA',
 'CO',
 'CT',
 'DC',
 'DE',
 'FL',
 'GA',
 'HI',
 'IA',
 'ID',
 'IL',
 'IN',
 'KA',
 'KS',
 'KY',
 'LA',
 'MA',
 'MD',
 'ME',
 'MI',
 'MN',
 'MO',
 'MS',
 'MT',
 'NC',
 'ND',
 'NE',
 'NH',
 'NJ',
 'NM',
 'NV',
 'NY',
 'OH',
 'OK',
 'OR',
 'PA',
 'PR',
 'RI',
 'SC',
 'SD',
 'TN',
 'TX',
 'UT',
 'VA',
 'VI',
 'VT',
 'WA',
 'WI',
 'WV',
 'WY',
 'ma'}

In [352]:
print((len(intlList)/len(pa2019))*100)
"The percentage about is how many international owners are present in the property assessments for the City of Boston for 2019. Solving this problem is a lot lower on the list than standardizing how Boston is recorded."

0.08702223647147732


'The percentage about is how many international owners are present in the property assessments for the City of Boston for 2019. Solving this problem is a lot lower on the list than standardizing how Boston is recorded.'

In [359]:
#add columns for city, state/region, and country
pa2019_intl['OwnerMailingAddressCity'] = 'International'
pa2019_intl['OwnerMailingAddressRegionState'] = 'International'
pa2019_intl['OwnerMailingAddressCountry'] = 'International'

pa2019_intl

,ParcelID,CondoMainID,GeographicInformationSystemID,StreetNumber,StreetName,StreetSuffix,UnitNumber,ZipCode,LandUse,OwnerOccupied,...,BuildingValue,TotalValue,GrossTax,LandSquareFoot,GrossArea,LivingArea,Fiscal_Year,OwnerMailingAddressCity,OwnerMailingAddressRegionState,OwnerMailingAddressCountry
5893,104727000,104727000.0,104727000,184,WEBSTER,ST,NaN,2128.0,CM,N,...,0,0,0,1817.0,0.0,0.0,2019,International,International,International
13957,203505320,203505100.0,203505100,197,EIGHTH,ST,PH12,2129.0,CD,N,...,1274100,1274100,1342901,1580.0,1580.0,1580.0,2019,International,International,International
14100,203506108,203506010.0,203506010,42,EIGHTH,ST,1407,2129.0,CD,N,...,421400,421400,444156,687.0,687.0,687.0,2019,International,International,International
14126,203506160,203506010.0,203506010,42,EIGHTH,ST,1521,2129.0,CD,N,...,867700,867700,914556,1383.0,1383.0,1383.0,2019,International,International,International
14391,203506726,203506010.0,203506010,42,EIGHTH,ST,5403,2129.0,CD,N,...,484200,484200,510347,681.0,681.0,681.0,2019,International,International,International
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35900,401404180,401404000.0,401404000,12,STONEHOLM,ST,526,2115.0,CD,N,...,348300,348300,367108,381.0,381.0,381.0,2019,International,International,International
48096,503006000,NaN,503006000,337,NEWBURY,ST,NaN,2115.0,C,N,...,1445900,3753500,9383750,2464.0,5808.0,3872.0,2019,International,International,International
50077,503558012,503558000.0,503558000,336,MARLBOROUGH,ST,6,2115.0,CD,N,...,358300,358300,377648,330.0,330.0,330.0,2019,International,International,International
51737,503807952,503807000.0,503807000,425,NEWBURY,ST,NaN,2115.0,CP,N,...,108000,108000,113832,160.0,0.0,0.0,2019,International,International,International
